<a href="https://colab.research.google.com/github/jonhrh1348/aviation-delay/blob/feature%2Fapi_data_setup/00_Initial_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from src.utils.config import *

In [ ]:
# 1. Start the Clickhouse Client for db use
ch_user = os.getenv('CLICKHOUSE_USER')
ch_password = os.getenv('CLICKHOUSE_PW')
ch_host = os.getenv('CLICKHOUSE_HOST')

client = setup_clickhouse_client(ch_user, ch_password, ch_host)
print('Client connected')

In [ ]:
# table schema setup

def create_table_if_not_exists(client, table_name, schema, index=None, partition=None, engine= "MergeTree()"):
    """
    Create a ClickHouse table if it doesn't exist.
    
    Args:
        client: ClickHouse client instance
        table_name: Name of the table to create
        schema: Dictionary mapping column names to their definitions
        engine: Storage engine (default: MergeTree)
    
    Returns:
        bool: True if table was created, False if it already existed
    """
    
    try:
        client.command(f"DROP TABLE IF EXISTS {table_name} SYNC")
    except Exception as e:
        print(f"Warning: Could not drop existing table: {e}")
        raise
    
    if table_name == 'raw_aviation_flights' or table_name == 'raw_hist_weather_data':
        # Build column definitions
        columns = ",\n    ".join([f"{col} {defn}" for col, defn in schema.items()])
        
        # Build and execute CREATE TABLE statement
        create_sql = f"""
            CREATE TABLE IF NOT EXISTS {table_name} (
                {columns}
            ) ENGINE = {engine}
            PRIMARY KEY (id);
        """
    elif table_name == 'aviation_flights' or table_name == 'historical_flight_data':
        columns = ",\n    ".join([f"{col} {defn}" for col, defn in schema.items()])
        
        create_sql = f"""
            CREATE TABLE IF NOT EXISTS {table_name} (
                {columns}
                {index}
            ) ENGINE = {engine}
            PARTITION BY toYYYYMM({partition})
            ORDER BY();
        """

    try:
        client.command(create_sql)
        print(f"Table '{table_name}' created successfully.")

    except Exception as e:
        print(f"Error creating table '{table_name}': {e}")
        raise


In [ ]:
# raw_aviation_data table & raw_hist_weather_data table creation
 
raw_av_schema = {
     'id': 'UUID DEFAULT generateUUIDv4()',
     'insert_time': "DateTime64(3, 'Asia/Singapore') DEFAULT now64()",
     'type': 'String',
     'status': 'String',
     'departure': 'JSON',
     'arrival': 'JSON',
     'airline': 'Map(String, String)',
     'flight': 'Map(String, String)',
     'codeshared_airline': 'Map(String, String)',
     'codeshared_flight': 'Map(String, String)'
     }
create_table_if_not_exists(client, 'raw_aviation_flights', raw_av_schema)

raw_we_schema = {
    'id': 'UUID DEFAULT generateUUIDv4()',
    'insert_time': "DateTime64(3, 'Asia/Singapore') DEFAULT now64()",
    'weather_time': 'DateTime',
    'main': 'JSON',
    'wind': 'JSON',
    'clouds': 'JSON',
    'rain': 'JSON',
    'snow': 'JSON',
    'weather': 'Array(JSON)'
}
create_table_if_not_exists(client, 'raw_hist_weather_data', raw_we_schema)

# historical_flight_data & aviation_flights tables creation

av_flight_schema = {
    'id': 'UUID DEFAULT generateUUIDv4()',
    'insert_time': "DateTime64(3, 'Asia/Singapore') DEFAULT now64()",
    'flight_type': 'String',
    'status': 'String',
    'iata_number': 'String',
    'airline_name': 'String',
    'dep_scheduled_time': 'DateTime64(3)',
    'dep_actual_time': 'DateTime64(3)',
    'dep_delay_mins': 'Int32',
    'arr_scheduled_time': 'DateTime64(3)',
    'arr_actual_time': 'DateTime64(3)',
    'arr_delay_mins': 'Int32',
    'codeshared_airline': 'String',
    'codeshared_flight': 'String'
} 
create_table_if_not_exists(client, 'aviation_flights', av_flight_schema, 
        "INDEX idx_status status TYPE set(10) GRANULARITY 4, \n" \
        "INDEX idx_airline_name airline_name TYPE bloom_filter(0.01) GRANULARITY 4", 
        "dep_scheduled_time")

hist_we_schema ={
    'id': 'UUID DEFAULT generateUUIDv4()',
    'insert_time': "DateTime64(3, 'Asia/Singapore') DEFAULT now64()",
    'date_observed': 'DateTime64(3)',
    'current_temp': 'Decimal32(2)',
    'feels_like_temp': 'Decimal32(2)',
    'pressure_hpa': 'Int32',
    'humidity_pct': 'Int16',
    'min_temp': 'Decimal32(2)',
    'max_temp': 'Decimal32(2)',
    'wind_speed_ms': 'Decimal32(2)',
    'wind_deg': 'Int16',
    'cloudiness_pct': 'Int16',
    'rain_1h': 'Decimal32(2)',
    'rain_3h': 'Decimal32(2)',
    'weather_main': 'Array(String)',
    'weather_desc': 'Array(String)'
}
create_table_if_not_exists(client, 'historical_weather_data', hist_we_schema, 
        "INDEX idx_current_temp current_temp TYPE minmax GRANULARITY 4, \n" \
        "INDEX idx_date_observed date_observed TYPE minmax GRANULARITY 4", 
        "date_observed")